# 3D Attention U-Net for Pancreas Segmentation in CT Scans

**Target Research Paper:**  
> *"Deep Learning Model with Attention Mechanism for a 3D Pancreas Segmentation in CT Scans"*  
> Mathematics 2025, 13, 3942 — DOI: [10.3390/math13243942](https://doi.org/10.3390/math13243942)  
> Authors: Tondji, Scapicchio, Lizzi, Fantacci, Oliva, Retico  
> Repository: [https://github.com/antonioroger2/Pancreas-Seg-MIP](https://github.com/antonioroger2/Pancreas-Seg-MIP)

### Target Environment & Dataset Constraints
- **Hardware:** Google Colab Tesla T4 GPU (~14.5 GB VRAM)
- **Dataset:** NIH Pancreas-CT (80 verified cases in `/content/drive/MyDrive/Pancreas-CT`)
- **Source Data:** Native CT volumes in `/content/drive/MyDrive/Pancreas-CT/Processed_data/` (Preserved and untouched)
- **2025 Preprocessed Destination:** `/content/drive/MyDrive/Pancreas-CT/2025_Processed_data/`
- **Methodology:** HU clip `[-100, 240]` → normalize `[0, 1]` → resample `1×1×1 mm³` isotropic → geometric center-crop `(224, 224, 128)`

---
## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Enable auto-reload for local code synchronization
# %load_ext autoreload
# %autoreload 2

: 

: 

---
## 2. Verify GPU & VRAM Availability

In [ ]:
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU Detected:    {gpu_name}")
    print(f"Total VRAM:      {total_vram:.2f} GB")
else:
    print("[WARNING] No GPU detected. Please enable GPU acceleration in Runtime -> Change runtime type.")

---
## 3. Install Required Dependencies

In [ ]:
!pip install -q monai nibabel SimpleITK tqdm matplotlib scipy pandas
print("[OK] Dependencies installed successfully.")

---
## 4. Deploy Repository Code from Zip & Configure Python Path

Extracts the authoritative local codebase from the uploaded zip file into `/content/drive/MyDrive/Pancreas-Seg-MIP/`.

**Re-run this cell** after uploading a new `pancreas_seg_code.zip` to pick up local changes.

In [ ]:
import os
import sys
import zipfile

# --- Configuration ---
REPO_DIR = '/content/drive/MyDrive/Pancreas-Seg-MIP'
ZIP_PATH = '/content/drive/MyDrive/pancreas_seg_code.zip'
SRC_DIR = os.path.join(REPO_DIR, 'src')

# --- Step 1: Extract zip if available ---
if os.path.exists(ZIP_PATH):
    os.makedirs(REPO_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(REPO_DIR)
    zip_size = os.path.getsize(ZIP_PATH) / 1024
    print(f'[OK] Extracted {ZIP_PATH} ({zip_size:.1f} KB) -> {REPO_DIR}')
elif os.path.exists(os.path.join(SRC_DIR, 'config.py')):
    print(f'[OK] Repository already exists at {REPO_DIR} (no zip found, using existing)')
else:
    raise FileNotFoundError(
        f'No code zip found at {ZIP_PATH} and no existing repo at {REPO_DIR}.\n'
        f'Upload pancreas_seg_code.zip to Google Drive root (My Drive) first.'
    )

# --- Step 2: Verify critical files exist ---
required_files = ['config.py', 'utils.py', 'train.py',
                  'models/model.py', 'models/attention.py',
                  'data/preprocessing.py', 'data/prepare_data.py']
missing = [f for f in required_files if not os.path.exists(os.path.join(SRC_DIR, f))]
if missing:
    raise FileNotFoundError(f'Missing critical files in {SRC_DIR}: {missing}')

# --- Step 3: Set working directory and Python path ---
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

py_files = sorted([os.path.relpath(os.path.join(r, f), SRC_DIR)
                   for r, _, fs in os.walk(SRC_DIR) for f in fs if f.endswith('.py')])
print(f'[OK] Working directory: {os.getcwd()}')
print(f'[OK] Python files ({len(py_files)}): {py_files}')

---
## 5. Define Canonical Dataset Paths & Import Modules

In [ ]:
import os
import glob
import numpy as np
import nibabel as nib
import SimpleITK as sitk
import pandas as pd
import matplotlib.pyplot as plt

# Import existing project modules
from src.config import HU_MIN, HU_MAX, TARGET_SPACING, CROP_SIZE, BATCH_SIZE, LEARNING_RATE
from src.models.model import build_model, count_parameters
from src.models.attention import AttentionGate3D
from src.losses import DiceFocalLoss
from src.data.dataset import PancreasVolumeDataset, discover_data_paths, create_fold_dataloaders
from src.data.preprocessing import preprocess_volume
from src.data.prepare_data import inspect_single_volume
from src.utils import memory_benchmark, set_seed, plot_3d_volume_slices

# Canonical Google Drive Paths
BASE = "/content/drive/MyDrive/Pancreas-CT"
RAW_DATA_DIR = os.path.join(BASE, "Processed_data")
PREPROCESSED_DIR = os.path.join(BASE, "2025_Processed_data")
CHECKPOINT_DIR = os.path.join(BASE, "checkpoints")
RESULTS_DIR = os.path.join(BASE, "results")
SPLITS_DIR = os.path.join(BASE, "splits")
MANIFEST_PATH = os.path.join(BASE, "2025_PREPROCESSING_MANIFEST.csv")
SUMMARY_PATH = os.path.join(BASE, "2025_PREPROCESSING_SUMMARY.md")

print(f"BASE Directory:            {BASE}")
print(f"Raw Input Source:          {RAW_DATA_DIR}")
print(f"2025 Target Preprocessed:  {PREPROCESSED_DIR}")
print(f"Checkpoints Directory:     {CHECKPOINT_DIR}")
print(f"Splits Directory:          {SPLITS_DIR}")

---
## 6. Verify Dataset: 80 Matched Pairs in Source Directory

Audit `Processed_data` to ensure exactly 80 images and 80 labels exist with identical patient IDs.

In [ ]:
import os
import glob
import re

raw_images = sorted(
    glob.glob(os.path.join(RAW_DATA_DIR, "images", "*.nii.gz"))
)

raw_labels = sorted(
    glob.glob(os.path.join(RAW_DATA_DIR, "labels", "*.nii.gz"))
)

print(f"Found {len(raw_images)} CT images and {len(raw_labels)} segmentation labels.")

assert len(raw_images) == 80, f"Expected 80 images, found {len(raw_images)}"
assert len(raw_labels) == 80, f"Expected 80 labels, found {len(raw_labels)}"


def extract_patient_id(path):
    """
    Extract patient identifier from filenames such as:
        PANCREAS_0001.nii.gz
        PANCREAS_0082.nii.gz
    """
    filename = os.path.basename(path)
    match = re.search(r"PANCREAS_(\d+)", filename, re.IGNORECASE)

    if match is None:
        raise ValueError(f"Could not extract patient ID from: {filename}")

    return f"PANCREAS_{match.group(1).zfill(4)}"


image_ids = {extract_patient_id(p) for p in raw_images}
label_ids = {extract_patient_id(p) for p in raw_labels}

missing_labels = sorted(image_ids - label_ids)
missing_images = sorted(label_ids - image_ids)

print(f"Unique image patient IDs: {len(image_ids)}")
print(f"Unique label patient IDs: {len(label_ids)}")

print(f"Images without labels: {missing_labels}")
print(f"Labels without images: {missing_images}")

assert not missing_labels, f"Missing labels for: {missing_labels}"
assert not missing_images, f"Missing images for: {missing_images}"

assert image_ids == label_ids, "Image and label patient ID sets do not match."

print("\n[OK] All 80 image/label pairs are verified and matched by Patient ID.")

---
## 7. Verify One Raw Patient Volume (PANCREAS_0001)

Inspect spatial metadata, intensity range, label uniqueness, and affine alignment before processing.

In [ ]:
p1_img = os.path.join(RAW_DATA_DIR, 'images', 'PANCREAS_0001.nii.gz')
p1_lbl = os.path.join(RAW_DATA_DIR, 'labels', 'PANCREAS_0001.nii.gz')

raw_meta = inspect_single_volume(p1_img, p1_lbl)

---
## 8. Run 2025 Paper Batch Preprocessing

Executes the full 80-case sequential preprocessing pipeline:
- Reads from `RAW_DATA_DIR` (`Processed_data` is strictly untouched)
- Writes compressed NIfTI files to `PREPROCESSED_DIR` (`2025_Processed_data`)
- Generates `2025_PREPROCESSING_MANIFEST.csv` and `2025_PREPROCESSING_SUMMARY.md`
- Validates exact foreground retention and geometric alignment per volume

In [ ]:
!python -m src.data.prepare_data \
    --preprocess_all \
    --raw_dir "{RAW_DATA_DIR}" \
    --preprocessed_dir "{PREPROCESSED_DIR}" \
    --manifest "{MANIFEST_PATH}" \
    --summary "{SUMMARY_PATH}"

---
## 9. Dedicated Post-Preprocessing Verification & Manifest Audit

Inspect output volume count, manifest records, retention statistics, and shape/spacing distributions.

In [ ]:
import os
import pandas as pd

BASE = "/content/drive/MyDrive/Pancreas-CT"
OUT = os.path.join(BASE, "2025_Processed_data")

print("Images:", len(os.listdir(os.path.join(OUT, "images"))))
print("Labels:", len(os.listdir(os.path.join(OUT, "labels"))))

manifest = os.path.join(BASE, "2025_PREPROCESSING_MANIFEST.csv")
df = pd.read_csv(manifest)

print("\nManifest cases:", len(df))

print("\nStatus:")
print(df["status"].value_counts())

print("\nRetention:")
print(df["foreground_retention_percent"].describe())

print("\nShapes:")
print(df["final_shape"].value_counts())

print("\nSpacing:")
print(df["final_spacing"].value_counts())

---
## 10. Multi-Slice Visual Inspection of Preprocessed Volume

In [ ]:
# Load preprocessed PANCREAS_0001 volume and visualize
sample_img_p = os.path.join(PREPROCESSED_DIR, 'images', 'PANCREAS_0001.nii.gz')
sample_lbl_p = os.path.join(PREPROCESSED_DIR, 'labels', 'PANCREAS_0001.nii.gz')

sample_img = nib.load(sample_img_p).get_fdata()
sample_lbl = nib.load(sample_lbl_p).get_fdata()

plot_3d_volume_slices(
    image_vol=sample_img,
    mask_vol=sample_lbl,
    pred_vol=None,
    save_path=os.path.join(BASE, 'pancreas_0001_preprocessed_slices.png'),
    n_slices=6,
    title='PANCREAS_0001 2025 Preprocessed (224x224x128)'
)

# Display plot
from IPython.display import Image, display
display(Image(os.path.join(BASE, 'pancreas_0001_preprocessed_slices.png')))

---
## 11. Generate Patient-Level 5-Fold CV + Independent Test Splits

Creates deterministic, reproducible patient-level splits saved to `splits/patient_splits.json`:
- 16 cases held out for independent testing
- 64 cases partitioned into 5 cross-validation folds (~13 per fold)

In [ ]:
from src.cross_validation import create_patient_splits, load_splits

img_paths, lbl_paths = discover_data_paths(PREPROCESSED_DIR)
n_patients = len(img_paths)

splits_file = os.path.join(SPLITS_DIR, 'patient_splits.json')
if os.path.exists(splits_file):
    splits = load_splits(SPLITS_DIR)
else:
    splits = create_patient_splits(n_patients, splits_dir=SPLITS_DIR)

print(f"\n[Splits Audit]")
print(f"  Total Patients:        {splits['n_patients']}")
print(f"  Independent Test Set:  {len(splits['test_indices'])} patients")
for f in splits['folds']:
    print(f"  Fold {f['fold']}: train = {len(f['train_indices'])} patients, val = {len(f['val_indices'])} patients")

---
## 12. Run Repository Integration Test Suite (12/12 Tests)

In [ ]:
!python -m src.test_all

---
## 13. Build 3D Attention U-Net Model & Parameter Audit

In [ ]:
model = build_model()
params = count_parameters(model)

print("=" * 60)
print("3D ATTENTION U-NET (Mathematics 2025 Paper Architecture)")
print("=" * 60)
print(f"  Encoder Channels:  16 -> 32 -> 64 -> 128 -> 256 (Figure 1)")
print(f"  Decoder Channels:  128 -> 64 -> 32 -> 16")
print(f"  Attention Gates:   4 Additive Attention Modules with LayerNorm")
print(f"  Total Parameters:  {params['total']:,} ({params['total_MB']:.2f} MB)")
print(f"  Trainable Params:  {params['trainable']:,}")
print("=" * 60)

---
## 14. Run Tesla T4 GPU Memory Benchmark

In [ ]:
bench = memory_benchmark()
if bench and bench.get('fits'):
    print("\n[OK] Model successfully fits inside Tesla T4 GPU memory with AMP.")

---
## 15. Real-Data One-Batch Forward/Backward Integration Test

Verifies that a real preprocessed 3D volume tensor flows through forward pass, `DiceFocalLoss`, backward gradient calculation, and Adam optimizer step without NaN or out-of-memory errors.

In [ ]:
import torch
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_model().to(device)
criterion = DiceFocalLoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=6e-4)
scaler = torch.amp.GradScaler(device.type, enabled=torch.cuda.is_available())

# Load 1 real volume from preprocessed dataset
img_paths, lbl_paths = discover_data_paths(PREPROCESSED_DIR)
real_dataset = PancreasVolumeDataset(img_paths[:1], lbl_paths[:1], augment=False, preprocessed=True)
real_loader = DataLoader(real_dataset, batch_size=1)

real_img, real_lbl = next(iter(real_loader))
real_img, real_lbl = real_img.to(device), real_lbl.to(device)

model.train()
optimizer.zero_grad(set_to_none=True)

with torch.amp.autocast(device_type=device.type, enabled=torch.cuda.is_available()):
    output = model(real_img)
    loss = criterion(output, real_lbl)

scaler.scale(loss).backward()
scaler.step(optimizer)
scaler.update()

print(f"Real Volume Input Shape:  {real_img.shape}")
print(f"Model Output Logits:      {output.shape}")
print(f"Dice + Focal Loss Value:  {loss.item():.6f}")
print("\n[OK] Real-data forward, backward, and optimizer step completed successfully!")

---
## 16. Model Training (Fold by Fold, Resume-Safe)

> **CRITICAL SAFETY NOTE:** Do NOT start training automatically until all preprocessing, splits, and memory benchmarks are audited and confirmed.
>
> Training is fully resumable: if Google Colab disconnects, re-executing this cell automatically resumes training from `latest_checkpoint.pth`.

In [ ]:
# Execute Training on Fold 0 (Change --fold to train other folds)
!python -m src.cross_validation \
    --data_dir "{PREPROCESSED_DIR}" \
    --checkpoint_dir "{CHECKPOINT_DIR}" \
    --drive_checkpoint_dir "{CHECKPOINT_DIR}" \
    --splits_dir "{SPLITS_DIR}" \
    --fold 0 \
    --epochs 300 \
    --batch_size 1 \
    --lr 6e-4

---
## 17. Quantitative Evaluation (Volumetric DSC, ASSD, HD95)

Evaluates trained checkpoints on validation sets and the independent test set.

In [ ]:
# Run volumetric evaluation
!python -m src.evaluate \
    --data_dir "{PREPROCESSED_DIR}" \
    --checkpoint_dir "{CHECKPOINT_DIR}" \
    --results_dir "{RESULTS_DIR}" \
    --splits_dir "{SPLITS_DIR}" \
    --fold 0